# 01 — Data preparation and cohort selection

## Purpose

This notebook prepares a historical Lending Club cohort for credit-risk research. It replaces the earlier approximately 10,000-loan snapshot with the larger accepted-loans dataset covering 2007–2018 Q4. The objective is to define an observed loan-outcome target, select a cohort with sufficiently mature outcomes, audit data quality, and save reusable tables for subsequent notebooks.

## Data locations

- Raw source used: `data/raw/accepted_2007_to_2018Q4.csv.gz`. This is an untouched compressed copy of the accepted-loans file downloaded through KaggleHub.
- KaggleHub also stored its original download in a local cache. The cache is not the input required by later notebooks.
- Outputs: `data/processed/pd_cohort_2011_2015_36m.parquet` and `data/processed/loss_outcomes_2011_2015_36m.parquet`.
- The processed files are separate from the raw source. Reproducing them requires running this notebook; notebooks 02 and 03 read the processed PD file instead.

## Cohort and target

The raw accepted-loans file contains 2,260,701 rows and 151 columns. This notebook selects 36-month loans issued from January 2011 through December 2015 whose recorded final status is either `Fully Paid` or `Charged Off`. The binary target `charged_off` is 1 for `Charged Off` and 0 for `Fully Paid`.

This target means **observed final charge-off versus full repayment**. It does not mean default within a fixed 12-month horizon. Loans marked `Current`, late, in a grace period, or otherwise unresolved are not assigned a negative label.

## Workflow

The notebook inspects the source schema, audits loan statuses by issue year and term, checks cohort resolution by month, examines missingness and suspicious values, and verifies unique loan IDs. It then saves an application/credit-data table for PD research and a separate repayment-outcome table for possible later loss analysis. Post-origination repayment amounts are not placed in the PD table.

Download data from kaggle

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wordsforthewise/lending-club")

print("Path to dataset files:", path)

100%|██████████| 1.26G/1.26G [02:31<00:00, 8.93MB/s]

Extracting files...


Path to dataset files: C:\Users\Yahya\.cache\kagglehub\datasets\wordsforthewise\lending-club\versions\3


Inspect the downloaded files

In [3]:
from pathlib import Path

dataset_dir = Path(path).resolve()

print("Dataset directory:", dataset_dir)
print("Directory exists:", dataset_dir.is_dir())

files = sorted(p for p in dataset_dir.rglob("*") if p.is_file())

print("\nNumber of files:", len(files))
print("Total file size (GiB):", round(
    sum(p.stat().st_size for p in files) / (1024**3), 2
))

print("\nFiles:")
for p in files:
    size_mb = p.stat().st_size / (1024**2)
    print(f"{p.relative_to(dataset_dir)}  |  {size_mb:,.1f} MB")

Dataset directory: C:\Users\Yahya\.cache\kagglehub\datasets\wordsforthewise\lending-club\versions\3
Directory exists: True

Number of files: 4
Total file size (GiB): 3.82

Files:
accepted_2007_to_2018q4.csv\accepted_2007_to_2018Q4.csv  |  1,597.5 MB
accepted_2007_to_2018Q4.csv.gz  |  374.4 MB
rejected_2007_to_2018q4.csv\rejected_2007_to_2018Q4.csv  |  1,699.7 MB
rejected_2007_to_2018Q4.csv.gz  |  243.6 MB


Copy the accepted gzip to the project data/raw folder and verify it.

In [4]:
import hashlib
import shutil

PROJECT_ROOT = Path.cwd().resolve().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Notebook working directory:", Path.cwd().resolve())
print("Proposed project root:    ", PROJECT_ROOT)

if not (PROJECT_ROOT / "data").is_dir():
    raise RuntimeError(
        "Proposed project root has no data/ folder. "
        "Set PROJECT_ROOT explicitly before copying."
    )

source = (
    Path(path)
    / "accepted_2007_to_2018Q4.csv.gz"
)
destination = RAW_DIR / source.name

if not source.is_file():
    raise FileNotFoundError(source)
if destination.exists():
    raise FileExistsError(
        f"Destination already exists; refusing to overwrite: {destination}"
    )

RAW_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(source, destination)

def sha256_file(file_path):
    digest = hashlib.sha256()
    with file_path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

source_hash = sha256_file(source)
destination_hash = sha256_file(destination)

print("Raw file:", destination)
print("Size (MB):", round(destination.stat().st_size / (1024**2), 1))
print("Hashes match:", source_hash == destination_hash)

if source_hash != destination_hash:
    raise RuntimeError("Copy verification failed. Do not delete the source.")

Notebook working directory: C:\Users\Yahya\Desktop\dynamic-credit-risk-xai\notebooks-&-codes
Proposed project root:     C:\Users\Yahya\Desktop\dynamic-credit-risk-xai
Raw file: C:\Users\Yahya\Desktop\dynamic-credit-risk-xai\data\raw\accepted_2007_to_2018Q4.csv.gz
Size (MB): 374.4
Hashes match: True


Inspect the raw file

In [5]:
import pandas as pd

raw_file = (
    Path.cwd().resolve().parent
    / "data"
    / "raw"
    / "accepted_2007_to_2018Q4.csv.gz"
)

if not raw_file.is_file():
    raise FileNotFoundError(raw_file)

columns = pd.read_csv(raw_file, nrows=0).columns.tolist()
preview = pd.read_csv(raw_file, nrows=5, low_memory=False)

print("Raw file:", raw_file)
print("Number of columns:", len(columns))
print("\nColumn names:")
print(columns)

print("\nFirst five rows—selected fields:")
fields_to_show = [
    "id", "issue_d", "term", "loan_status", "loan_amnt",
    "int_rate", "annual_inc", "dti", "grade",
    "total_pymnt", "recoveries"
]
available = [name for name in fields_to_show if name in preview.columns]
print(preview[available].to_string(index=False))

print("\nPreview data types:")
print(preview[available].dtypes.to_string())

Raw file: C:\Users\Yahya\Desktop\dynamic-credit-risk-xai\data\raw\accepted_2007_to_2018Q4.csv.gz
Number of columns: 151

Column names:
['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_sinc

Outcome & vintage audit

In [6]:
audit_parts = []

for chunk in pd.read_csv(
    raw_file,
    usecols=["issue_d", "term", "loan_status"],
    chunksize=200_000,
    compression="infer",
    low_memory=False,
):
    chunk["issue_year"] = pd.to_datetime(
        chunk["issue_d"], format="%b-%Y", errors="coerce"
    ).dt.year
    chunk["term_months"] = pd.to_numeric(
        chunk["term"].astype("string").str.extract(r"(\d+)", expand=False),
        errors="coerce",
    )

    counts = (
        chunk.groupby(
            ["issue_year", "term_months", "loan_status"],
            dropna=False,
        )
        .size()
        .rename("loans")
        .reset_index()
    )
    audit_parts.append(counts)

audit = (
    pd.concat(audit_parts, ignore_index=True)
    .groupby(
        ["issue_year", "term_months", "loan_status"],
        dropna=False,
    )["loans"]
    .sum()
    .reset_index()
)

print("Total rows:", f"{audit['loans'].sum():,}")

print("\nLoan statuses:")
print(
    audit.groupby("loan_status", dropna=False)["loans"]
    .sum()
    .sort_values(ascending=False)
    .to_string()
)

print("\nIssue years:")
print(
    audit.groupby("issue_year", dropna=False)["loans"]
    .sum()
    .sort_index()
    .to_string()
)

print("\nLoan terms:")
print(
    audit.groupby("term_months", dropna=False)["loans"]
    .sum()
    .sort_index()
    .to_string()
)

print("\nStatus by issue year:")
print(
    audit.pivot_table(
        index="issue_year",
        columns="loan_status",
        values="loans",
        aggfunc="sum",
        fill_value=0,
    ).to_string()
)

Total rows: 2,260,701

Loan statuses:
loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
NaN                                                         33

Issue years:
issue_year
2007.0       603
2008.0      2393
2009.0      5281
2010.0     12537
2011.0     21721
2012.0     53367
2013.0    134814
2014.0    235629
2015.0    421095
2016.0    434407
2017.0    443579
2018.0    495242
NaN           33

Loan terms:
term_months
36      1609754
60       650914
<NA>         33



Resolution by year and term

In [7]:
resolved_statuses = {"Fully Paid", "Charged Off"}

cohort_audit = audit.copy()
cohort_audit["is_resolved"] = cohort_audit["loan_status"].isin(
    resolved_statuses
)

cohort_summary = (
    cohort_audit
    .assign(
        resolved_loans=lambda x: x["loans"].where(x["is_resolved"], 0),
        charged_off=lambda x: x["loans"].where(
            x["loan_status"].eq("Charged Off"), 0
        ),
    )
    .groupby(["issue_year", "term_months"], dropna=False)[
        ["loans", "resolved_loans", "charged_off"]
    ]
    .sum()
    .reset_index()
)

cohort_summary["resolved_pct"] = (
    100 * cohort_summary["resolved_loans"] / cohort_summary["loans"]
).round(1)

cohort_summary["charged_off_pct_among_resolved"] = (
    100
    * cohort_summary["charged_off"]
    / cohort_summary["resolved_loans"].replace(0, float("nan"))
).round(1)

print(
    cohort_summary[
        [
            "issue_year",
            "term_months",
            "loans",
            "resolved_loans",
            "resolved_pct",
            "charged_off",
            "charged_off_pct_among_resolved",
        ]
    ].to_string(index=False)
)

 issue_year  term_months  loans  resolved_loans  resolved_pct  charged_off  charged_off_pct_among_resolved
     2007.0           36    603             251          41.6           45                            17.9
     2008.0           36   2393            1562          65.3          247                            15.8
     2009.0           36   5281            4716          89.3          594                            12.6
     2010.0           36   9156            8466          92.5          842                             9.9
     2010.0           60   3381            3070          90.8          645                            21.0
     2011.0           36  14101           14101         100.0         1499                            10.6
     2011.0           60   7620            7620         100.0         1798                            23.6
     2012.0           36  43470           43470         100.0         5903                            13.6
     2012.0           60   9897      

Monthly cohort audit

In [8]:
monthly_parts = []

for chunk in pd.read_csv(
    raw_file,
    usecols=["issue_d", "term", "loan_status"],
    chunksize=200_000,
    compression="infer",
    low_memory=False,
):
    issue_month = pd.to_datetime(
        chunk["issue_d"], format="%b-%Y", errors="coerce"
    )
    term_months = pd.to_numeric(
        chunk["term"].astype("string").str.extract(r"(\d+)", expand=False),
        errors="coerce",
    )

    eligible = (
        term_months.eq(36)
        & issue_month.between("2011-01-01", "2015-12-01")
    )

    part = (
        pd.DataFrame({
            "issue_month": issue_month[eligible],
            "loan_status": chunk.loc[eligible, "loan_status"],
        })
        .groupby(["issue_month", "loan_status"], dropna=False)
        .size()
        .rename("loans")
        .reset_index()
    )
    monthly_parts.append(part)

monthly_audit = (
    pd.concat(monthly_parts, ignore_index=True)
    .groupby(["issue_month", "loan_status"], dropna=False)["loans"]
    .sum()
    .reset_index()
)

year_status = (
    monthly_audit
    .assign(issue_year=lambda x: x["issue_month"].dt.year)
    .pivot_table(
        index="issue_year",
        columns="loan_status",
        values="loans",
        aggfunc="sum",
        fill_value=0,
    )
)

print("Yearly status counts for proposed 36-month cohort:")
print(year_status.to_string())

print("\n2015 monthly status counts:")
print(
    monthly_audit[
        monthly_audit["issue_month"].dt.year.eq(2015)
    ]
    .pivot_table(
        index="issue_month",
        columns="loan_status",
        values="loans",
        aggfunc="sum",
        fill_value=0,
    )
    .to_string()
)

Yearly status counts for proposed 36-month cohort:
loan_status  Charged Off  Current  Fully Paid  In Grace Period  Late (16-30 days)  Late (31-120 days)
issue_year                                                                                           
2011                1499        0       12602                0                  0                   0
2012                5903        0       37567                0                  0                   0
2013               12378        0       88044                0                  0                   0
2014               22315        0      140255                0                  0                   0
2015               42132       72      240894                9                  5                  61

2015 monthly status counts:
loan_status  Charged Off  Current  Fully Paid  In Grace Period  Late (16-30 days)  Late (31-120 days)
issue_month                                                                                          
20

Inspect cohort data quality

In [9]:
inspection_columns = [
    "id", "issue_d", "term", "loan_status",
    "loan_amnt", "funded_amnt", "int_rate", "installment",
    "grade", "sub_grade", "emp_length", "home_ownership",
    "annual_inc", "verification_status", "purpose",
    "dti", "earliest_cr_line", "fico_range_low", "fico_range_high",
    "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec",
    "revol_bal", "revol_util", "total_acc",
    "total_rec_prncp", "recoveries", "collection_recovery_fee"
]

parts = []

for chunk in pd.read_csv(
    raw_file,
    usecols=inspection_columns,
    chunksize=100_000,
    compression="infer",
    low_memory=False,
):
    issue_month = pd.to_datetime(
        chunk["issue_d"], format="%b-%Y", errors="coerce"
    )
    term_months = pd.to_numeric(
        chunk["term"].astype("string").str.extract(r"(\d+)", expand=False),
        errors="coerce",
    )

    keep = (
        term_months.eq(36)
        & issue_month.between("2011-01-01", "2015-12-01")
        & chunk["loan_status"].isin(["Fully Paid", "Charged Off"])
    )

    selected = chunk.loc[keep].copy()
    selected["issue_month"] = issue_month.loc[keep]
    parts.append(selected)

cohort = pd.concat(parts, ignore_index=True)

print("Selected rows:", f"{len(cohort):,}")
print("Columns retained for inspection:", len(cohort.columns))
print("Memory used (MiB):", round(
    cohort.memory_usage(deep=True).sum() / (1024**2), 1
))
print("Duplicate non-null IDs:", cohort.loc[
    cohort["id"].notna(), "id"
].duplicated().sum())
print("Missing IDs:", cohort["id"].isna().sum())

print("\nMissing values (%), highest first:")
print(
    (cohort.isna().mean() * 100)
    .sort_values(ascending=False)
    .round(2)
    .to_string()
)

print("\nData types:")
print(cohort.dtypes.to_string())

Selected rows: 603,589
Columns retained for inspection: 30
Memory used (MiB): 438.8
Duplicate non-null IDs: 0
Missing IDs: 0

Missing values (%), highest first:
emp_length                 5.95
revol_util                 0.05
dti                        0.00
term                       0.00
funded_amnt                0.00
id                         0.00
installment                0.00
grade                      0.00
sub_grade                  0.00
home_ownership             0.00
annual_inc                 0.00
verification_status        0.00
int_rate                   0.00
loan_amnt                  0.00
loan_status                0.00
issue_d                    0.00
delinq_2yrs                0.00
purpose                    0.00
fico_range_low             0.00
fico_range_high            0.00
inq_last_6mths             0.00
earliest_cr_line           0.00
open_acc                   0.00
pub_rec                    0.00
revol_bal                  0.00
total_acc                  0.00
total_r

Values and category audit

In [10]:
numeric_cols = [
    "loan_amnt", "funded_amnt", "int_rate", "installment",
    "annual_inc", "dti", "fico_range_low", "fico_range_high",
    "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec",
    "revol_bal", "revol_util", "total_acc",
    "total_rec_prncp", "recoveries", "collection_recovery_fee",
]

print("NUMERIC SUMMARY")
print(
    cohort[numeric_cols]
    .describe(percentiles=[0.01, 0.50, 0.99])
    .T[["count", "min", "1%", "50%", "99%", "max"]]
    .round(2)
    .to_string()
)

print("\nCATEGORY COUNTS")
for col in [
    "grade", "home_ownership", "verification_status",
    "purpose", "emp_length", "loan_status"
]:
    print(f"\n{col}:")
    print(cohort[col].value_counts(dropna=False).to_string())

print("\nOUTCOMES BY ISSUE YEAR")
print(
    pd.crosstab(
        cohort["issue_month"].dt.year,
        cohort["loan_status"],
        margins=True,
    ).to_string()
)

print("\nBASIC INVALID-VALUE COUNTS")
checks = {
    "loan_amnt <= 0": cohort["loan_amnt"].le(0).sum(),
    "funded_amnt <= 0": cohort["funded_amnt"].le(0).sum(),
    "annual_inc < 0": cohort["annual_inc"].lt(0).sum(),
    "dti < 0": cohort["dti"].lt(0).sum(),
    "fico_range_low > fico_range_high": (
        cohort["fico_range_low"] > cohort["fico_range_high"]
    ).sum(),
    "total_rec_prncp < 0": cohort["total_rec_prncp"].lt(0).sum(),
    "recoveries < 0": cohort["recoveries"].lt(0).sum(),
}
print(pd.Series(checks).to_string())

NUMERIC SUMMARY
                            count      min        1%       50%        99%         max
loan_amnt                603589.0  1000.00   1500.00  10000.00   35000.00    35000.00
funded_amnt              603589.0  1000.00   1500.00  10000.00   35000.00    35000.00
int_rate                 603589.0     5.32      5.32     11.99      22.15       28.99
installment              603589.0    14.01     52.89    344.18    1207.30     1445.46
annual_inc               603589.0     0.00  17400.00  60000.00  250000.00  9000000.00
dti                      603587.0     0.00      1.99     17.16      37.15      999.00
fico_range_low           603589.0   660.00    660.00    685.00     795.00      845.00
fico_range_high          603589.0   664.00    664.00    689.00     799.00      850.00
delinq_2yrs              603589.0     0.00      0.00      0.00       4.00       30.00
inq_last_6mths           603589.0     0.00      0.00      0.00       4.00        8.00
open_acc                 603589.0     

Inspect suspicious values

In [11]:
checks = {
    "annual_inc == 0": cohort["annual_inc"].eq(0),
    "annual_inc > 1,000,000": cohort["annual_inc"].gt(1_000_000),
    "dti is missing": cohort["dti"].isna(),
    "dti >= 100": cohort["dti"].ge(100),
    "dti == 999": cohort["dti"].eq(999),
    "revol_util is missing": cohort["revol_util"].isna(),
    "revol_util > 100": cohort["revol_util"].gt(100),
    "funded_amnt > loan_amnt": (
        cohort["funded_amnt"] > cohort["loan_amnt"]
    ),
    "total_rec_prncp > funded_amnt": (
        cohort["total_rec_prncp"] > cohort["funded_amnt"]
    ),
    "recoveries > 0 and Fully Paid": (
        cohort["recoveries"].gt(0)
        & cohort["loan_status"].eq("Fully Paid")
    ),
}

print("POTENTIAL ISSUE COUNTS")
print(pd.Series({name: int(mask.sum()) for name, mask in checks.items()}).to_string())

show_cols = [
    "id", "issue_month", "loan_status", "loan_amnt",
    "annual_inc", "dti", "revol_util",
    "funded_amnt", "total_rec_prncp", "recoveries",
]

for name in [
    "annual_inc == 0",
    "annual_inc > 1,000,000",
    "dti >= 100",
    "revol_util > 100",
]:
    print(f"\nEXAMPLES: {name}")
    print(cohort.loc[checks[name], show_cols].head(5).to_string(index=False))

POTENTIAL ISSUE COUNTS
annual_inc == 0                     2
annual_inc > 1,000,000            105
dti is missing                      2
dti >= 100                          5
dti == 999                          1
revol_util is missing             307
revol_util > 100                 2203
funded_amnt > loan_amnt             0
total_rec_prncp > funded_amnt      45
recoveries > 0 and Fully Paid       0

EXAMPLES: annual_inc == 0
      id issue_month loan_status  loan_amnt  annual_inc  dti  revol_util  funded_amnt  total_rec_prncp  recoveries
66415476  2015-12-01  Fully Paid    20000.0         0.0  NaN        88.4      20000.0         20000.00         0.0
65199509  2015-11-01 Charged Off     3700.0         0.0  NaN        56.9       3700.0           258.05         0.0

EXAMPLES: annual_inc > 1,000,000
      id issue_month loan_status  loan_amnt  annual_inc  dti  revol_util  funded_amnt  total_rec_prncp  recoveries
68132140  2015-12-01  Fully Paid    16000.0   1400000.0 0.07         6.5    

Audit dates and split viability

In [12]:
yearly = (
    cohort.assign(
        issue_year=cohort["issue_month"].dt.year,
        charged_off=cohort["loan_status"].eq("Charged Off").astype(int),
    )
    .groupby("issue_year")
    .agg(
        loans=("id", "size"),
        charged_off=("charged_off", "sum"),
    )
)

yearly["charged_off_rate_pct"] = (
    100 * yearly["charged_off"] / yearly["loans"]
).round(2)

print("YEARLY OUTCOMES")
print(yearly.to_string())

split_rules = {
    "train_2011_2013": ("2011-01-01", "2013-12-01"),
    "validation_2014": ("2014-01-01", "2014-12-01"),
    "test_2015": ("2015-01-01", "2015-12-01"),
}

print("\nPROPOSED OUT-OF-TIME SPLITS")
for name, (start, end) in split_rules.items():
    subset = cohort.loc[cohort["issue_month"].between(start, end)]
    events = subset["loan_status"].eq("Charged Off").sum()
    print(
        f"{name}: {len(subset):,} loans | "
        f"{events:,} charged off | "
        f"{events / len(subset):.2%} charged-off rate"
    )

print("\nISSUE MONTH RANGE")
print("Earliest:", cohort["issue_month"].min())
print("Latest:  ", cohort["issue_month"].max())

YEARLY OUTCOMES
             loans  charged_off  charged_off_rate_pct
issue_year                                           
2011         14101         1499                 10.63
2012         43470         5903                 13.58
2013        100422        12378                 12.33
2014        162570        22315                 13.73
2015        283026        42132                 14.89

PROPOSED OUT-OF-TIME SPLITS
train_2011_2013: 157,993 loans | 19,780 charged off | 12.52% charged-off rate
validation_2014: 162,570 loans | 22,315 charged off | 13.73% charged-off rate
test_2015: 283,026 loans | 42,132 charged off | 14.89% charged-off rate

ISSUE MONTH RANGE
Earliest: 2011-01-01 00:00:00
Latest:   2015-12-01 00:00:00


Inspect the remaining columns

In [13]:
all_columns_sample = pd.read_csv(
    raw_file,
    nrows=100,
    compression="infer",
    low_memory=False,
)

not_yet_inspected = [
    col for col in all_columns_sample.columns
    if col not in inspection_columns
]

print("Columns not included in the first cohort load:",
      len(not_yet_inspected))

for col in not_yet_inspected:
    values = (
        all_columns_sample[col]
        .dropna()
        .astype(str)
        .unique()[:2]
        .tolist()
    )
    print(
        f"{col:<45} "
        f"dtype={str(all_columns_sample[col].dtype):<10} "
        f"examples={values}"
    )

Columns not included in the first cohort load: 122
member_id                                     dtype=float64    examples=[]
funded_amnt_inv                               dtype=float64    examples=['3600.0', '24700.0']
emp_title                                     dtype=object     examples=['leadman', 'Engineer']
pymnt_plan                                    dtype=object     examples=['n']
url                                           dtype=object     examples=['https://lendingclub.com/browse/loanDetail.action?loan_id=68407277', 'https://lendingclub.com/browse/loanDetail.action?loan_id=68355089']
desc                                          dtype=float64    examples=[]
title                                         dtype=object     examples=['Debt consolidation', 'Business']
zip_code                                      dtype=object     examples=['190xx', '577xx']
addr_state                                    dtype=object     examples=['PA', 'SD']
mths_since_last_delinq               

Audit remaining fields

In [14]:
additional_columns = [
    "id", "issue_d", "term", "loan_status",
    "application_type", "addr_state", "initial_list_status",
    "mths_since_last_delinq", "mths_since_last_record",
    "collections_12_mths_ex_med", "mths_since_last_major_derog",
    "acc_now_delinq", "tot_coll_amt", "tot_cur_bal",
    "open_acc_6m", "open_act_il", "total_bal_il", "il_util",
    "all_util", "total_rev_hi_lim", "inq_last_12m",
    "acc_open_past_24mths", "avg_cur_bal", "bc_util",
    "mort_acc", "num_accts_ever_120_pd",
    "pct_tl_nvr_dlq", "pub_rec_bankruptcies",
    "total_bal_ex_mort", "total_bc_limit",
    "annual_inc_joint", "dti_joint",
]

audit_parts_extra = []

for chunk in pd.read_csv(
    raw_file,
    usecols=additional_columns,
    chunksize=100_000,
    compression="infer",
    low_memory=False,
):
    issue_month = pd.to_datetime(
        chunk["issue_d"], format="%b-%Y", errors="coerce"
    )
    term_months = pd.to_numeric(
        chunk["term"].astype("string").str.extract(r"(\d+)", expand=False),
        errors="coerce",
    )

    keep = (
        term_months.eq(36)
        & issue_month.between("2011-01-01", "2015-12-01")
        & chunk["loan_status"].isin(["Fully Paid", "Charged Off"])
    )

    selected = chunk.loc[keep].copy()
    selected["issue_year"] = issue_month.loc[keep].dt.year
    audit_parts_extra.append(selected)

extra = pd.concat(audit_parts_extra, ignore_index=True)

print("Rows:", f"{len(extra):,}")
print("Duplicate non-null IDs:", extra["id"].dropna().duplicated().sum())
print("Unique non-null IDs:", extra["id"].nunique())

report_cols = [
    col for col in additional_columns
    if col not in {"id", "issue_d", "term", "loan_status"}
]

print("\nMISSING RATE BY YEAR (%):")
print(
    extra.groupby("issue_year")[report_cols]
    .apply(lambda group: group.isna().mean().mul(100).round(1))
    .T
    .to_string()
)

print("\nDISTINCT VALUES ACROSS SELECTED COHORT:")
print(
    extra[report_cols]
    .nunique(dropna=True)
    .sort_values()
    .to_string()
)

Rows: 603,589
Duplicate non-null IDs: 0
Unique non-null IDs: 603589

MISSING RATE BY YEAR (%):
issue_year                    2011   2012   2013   2014  2015
application_type               0.0    0.0    0.0    0.0   0.0
addr_state                     0.0    0.0    0.0    0.0   0.0
initial_list_status            0.0    0.0    0.0    0.0   0.0
mths_since_last_delinq        66.4   59.1   56.5   48.9  48.0
mths_since_last_record        94.9   97.3   87.4   81.1  81.3
collections_12_mths_ex_med     0.0    0.0    0.0    0.0   0.0
mths_since_last_major_derog  100.0   89.8   79.5   70.7  69.8
acc_now_delinq                 0.0    0.0    0.0    0.0   0.0
tot_coll_amt                 100.0   51.9    0.0    0.0   0.0
tot_cur_bal                  100.0   51.9    0.0    0.0   0.0
open_acc_6m                  100.0  100.0  100.0  100.0  94.7
open_act_il                  100.0  100.0  100.0  100.0  94.7
total_bal_il                 100.0  100.0  100.0  100.0  94.7
il_util                      100.0  1

Verify the join

In [15]:
extra_keep = [
    "id",
    "application_type",
    "addr_state",
    "initial_list_status",
    "collections_12_mths_ex_med",
    "acc_now_delinq",
    "pub_rec_bankruptcies",
]

assert cohort["id"].is_unique
assert extra["id"].is_unique
assert len(cohort) == len(extra) == 603_589

combined = cohort.merge(
    extra[extra_keep],
    on="id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("Rows before join:", len(cohort))
print("Rows after join: ", len(combined))
print("\nJoin result:")
print(combined["_merge"].value_counts(dropna=False).to_string())

print("\nApplication type:")
print(combined["application_type"].value_counts(dropna=False).to_string())

print("\nInitial list status:")
print(combined["initial_list_status"].value_counts(dropna=False).to_string())

print("\nMissing values in newly joined fields:")
print(
    combined[extra_keep[1:]]
    .isna()
    .sum()
    .to_string()
)

combined = combined.drop(columns="_merge")

Rows before join: 603589
Rows after join:  603589

Join result:
_merge
both          603589
left_only          0
right_only         0

Application type:
application_type
Individual    603350
Joint App        239

Initial list status:
initial_list_status
f    329147
w    274442

Missing values in newly joined fields:
application_type              0
addr_state                    0
initial_list_status           0
collections_12_mths_ex_med    0
acc_now_delinq                0
pub_rec_bankruptcies          0


Prepare

In [16]:
pd_columns = [
    "id", "issue_month", "loan_status",
    "loan_amnt", "funded_amnt", "term",
    "int_rate", "installment", "grade", "sub_grade",
    "emp_length", "home_ownership", "annual_inc",
    "verification_status", "purpose", "dti",
    "earliest_cr_line", "fico_range_low", "fico_range_high",
    "delinq_2yrs", "inq_last_6mths", "open_acc",
    "pub_rec", "revol_bal", "revol_util", "total_acc",
    "application_type", "addr_state", "initial_list_status",
    "collections_12_mths_ex_med", "acc_now_delinq",
    "pub_rec_bankruptcies",
]

loss_columns = [
    "id", "loan_amnt", "funded_amnt",
    "total_rec_prncp", "recoveries",
    "collection_recovery_fee",
]

pd_data = combined[pd_columns].copy()
pd_data["charged_off"] = (
    pd_data["loan_status"].eq("Charged Off").astype("int8")
)

loss_data = combined[loss_columns].copy()

print("PD rows/columns:", pd_data.shape)
print("Loss rows/columns:", loss_data.shape)
print("IDs unique in PD data:", pd_data["id"].is_unique)
print("IDs unique in loss data:", loss_data["id"].is_unique)

print("\nTarget counts:")
print(pd_data["charged_off"].value_counts().sort_index().to_string())

print("\nYears:")
print(pd_data["issue_month"].dt.year.value_counts().sort_index().to_string())

print("\nPD data: missing values > 0:")
print(
    pd_data.isna().sum()
    .loc[lambda s: s.gt(0)]
    .sort_values(ascending=False)
    .to_string()
)

print("\nLoss data: missing values > 0:")
missing_loss = loss_data.isna().sum()
print(missing_loss.loc[missing_loss.gt(0)].to_string()
      if missing_loss.gt(0).any() else "None")

print("\nPost-origination columns in PD data:")
post_origination = [
    "total_rec_prncp", "recoveries", "collection_recovery_fee",
    "total_pymnt", "out_prncp", "last_pymnt_amnt",
]
print([col for col in post_origination if col in pd_data.columns])

PD rows/columns: (603589, 33)
Loss rows/columns: (603589, 6)
IDs unique in PD data: True
IDs unique in loss data: True

Target counts:
charged_off
0    519362
1     84227

Years:
issue_month
2011     14101
2012     43470
2013    100422
2014    162570
2015    283026

PD data: missing values > 0:
emp_length    35885
revol_util      307
dti               2

Loss data: missing values > 0:
None

Post-origination columns in PD data:
[]


In [18]:
for name, table in [("PD", pd_data), ("loss", loss_data)]:
    print(f"\n{name} id dtype:", table["id"].dtype)
    print(
        "Python types:",
        table["id"].map(lambda value: type(value).__name__)
        .value_counts()
        .to_dict()
    )

for output in [pd_file, loss_file]:
    print(
        output.name,
        "| exists:", output.exists(),
        "| size (bytes):",
        output.stat().st_size if output.exists() else "—",
    )


PD id dtype: object
Python types: {'int': 413655, 'str': 189934}

loss id dtype: object
Python types: {'int': 413655, 'str': 189934}
pd_cohort_2011_2015_36m.parquet | exists: False | size (bytes): —
loss_outcomes_2011_2015_36m.parquet | exists: False | size (bytes): —


Save and verify

In [19]:
# IDs are labels, not numerical measurements.
pd_data["id"] = pd_data["id"].astype("string")
loss_data["id"] = loss_data["id"].astype("string")

assert pd_data["id"].notna().all()
assert loss_data["id"].notna().all()
assert pd_data["id"].is_unique
assert loss_data["id"].is_unique
assert len(pd_data) == len(loss_data) == 603_589
assert pd_data["id"].equals(loss_data["id"])
assert int(pd_data["charged_off"].sum()) == 84_227
assert int(loss_data["charged_off"].sum()) == 84_227

for output in (pd_file, loss_file):
    if output.exists():
        raise FileExistsError(f"Refusing to overwrite: {output}")

pd_data.to_parquet(pd_file, index=False, compression="snappy")
loss_data.to_parquet(loss_file, index=False, compression="snappy")

pd_check = pd.read_parquet(pd_file)
loss_check = pd.read_parquet(loss_file)

assert pd_check.shape == pd_data.shape
assert loss_check.shape == loss_data.shape
assert pd_check["id"].equals(loss_check["id"])
assert pd_check["id"].is_unique
assert int(pd_check["charged_off"].sum()) == 84_227

print("Saved and verified:")
for output in (pd_file, loss_file):
    print(f"{output.name}: {output.stat().st_size / (1024**2):.1f} MiB")

print("PD table:", pd_check.shape)
print("Loss table:", loss_check.shape)
print("Charged-off loans:", int(pd_check["charged_off"].sum()))
print("IDs match in row order:", pd_check["id"].equals(loss_check["id"]))

Saved and verified:
pd_cohort_2011_2015_36m.parquet: 16.9 MiB
loss_outcomes_2011_2015_36m.parquet: 7.9 MiB
PD table: (603589, 33)
Loss table: (603589, 9)
Charged-off loans: 84227
IDs match in row order: True


## Conclusion and handoff

The selected cohort contains **603,589** resolved 36-month loans: **519,362** `Fully Paid` and **84,227** `Charged Off`. Its issue dates span 2011–2015. The saved files were read back and verified after writing:

| File | Rows | Columns | Purpose |
|---|---:|---:|---|
| `data/processed/pd_cohort_2011_2015_36m.parquet` | 603,589 | 33 | Candidate PD inputs, identifiers, issue month, recorded status, and target |
| `data/processed/loss_outcomes_2011_2015_36m.parquet` | 603,589 | 9 | Loan and repayment-outcome amounts for later loss research |

Both tables have unique IDs, matching row-order IDs, and identical target totals. The PD table contains **no** `total_rec_prncp`, `recoveries`, or `collection_recovery_fee` columns. The loss table is an outcome-analysis source, not a PD feature table.

### What was excluded or deferred

- Rejected applications were not used because their eventual repayment outcomes are not observed in the same way as those of originated loans.
- Loans outside the selected 2011–2015, 36-month cohort were excluded from these processed outputs.
- Within that cohort, unresolved statuses—including 147 unresolved 2015 loans—were excluded rather than labelled non-default. Historical `Does not meet the credit policy` statuses were also outside the selected cohort.
- Most of the original 151 columns were **not carried into** these two narrowly defined outputs. Some were identifiers or post-origination fields; others were deferred because their availability varied greatly by year or their timing needed further investigation. This is a selected schema, not a claim that every omitted column was erroneous.
- No rows were removed solely for high income, high DTI, or revolving utilisation.

### Data-quality findings

There are no duplicate or missing IDs in the selected cohort. The PD table retains missing values in `emp_length` (35,885), `revol_util` (307), and `dti` (2). The audit also found two zero-income records, one DTI value of 999, and some extreme income and revolving-utilisation values. These were documented rather than broadly deleted or imputed. Their treatment as model inputs belongs in feature engineering.

### Limitations

The dataset is a historical snapshot, not a live feed. Selecting only loans with final outcomes can still introduce selection effects, particularly near the end of observation. The target is a final recorded status, not a time-to-default or fixed-horizon PD. Credit outcomes are observed only for loans that Lending Club originated; conclusions must not automatically be extended to rejected applicants. Repayment amounts have been retained for possible LGD work, but this notebook has **not** established or validated an LGD or EAD model.

**Next:** `02-EDA.ipynb` reads the saved PD Parquet file; it does not redownload the raw data.